In [ ]:
!pip install langchain_huggingface langchain_community pypdf langchain-text-splitters

In [ ]:
!pip install -q -U opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp-proto-grpc chromadb langchain-chroma

In [ ]:
import os
import urllib.request
import torch
import pandas as pd

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace, HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

## RAG 파이프라인

###데이터 다운로드

In [ ]:
# 분석할 PDF 파일을 웹에서 다운로드.
urllib.request.urlretrieve("https://github.com/llama-index-tutorial/llama-index-tutorial/raw/main/ch07/2023_%EB%B6%81%ED%95%9C%EC%9D%B8%EA%B6%8C%EB%B3%B4%EA%B3%A0%EC%84%9C.pdf", filename="2023_북한인권보고서.pdf")

###LLM 설정

In [ ]:
llm = ChatHuggingFace(llm=HuggingFacePipeline.from_model_id(
    model_id='Qwen/Qwen2.5-3B-Instruct',        # GPT-4o 대신 사용할 LLM
    task='text-generation',
    device=0,
    model_kwargs={'torch_dtype':torch.float16},
    pipeline_kwargs={
        'max_new_tokens': 500,
        'do_sample': False,
        'return_full_text': False,
    },
))

###임베딩 모델

In [ ]:
embed_model = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',                    # text-embedding-3-large 대신 사용할 임베딩
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True},
)

###문서 분할 설정

In [ ]:
# 문서 분할 설정
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,  # 문서를 300자 단위로 분할
    chunk_overlap=100,  # 문맥 유지를 위해 청크 간 100자 중복
)

###pdf load & embedding

In [ ]:
loader = PyPDFLoader("2023_북한인권보고서.pdf")  # PDF 문서 로더
documents = loader.load()  # 문서에서 텍스트 추출
chunks = text_splitter.split_documents(documents)  # 문서 분할

In [ ]:
# vector_store.delete_collection()
vector_store = Chroma.from_documents(chunks, embed_model)
print("벡터스토어 문서 수:", vector_store._collection.count())   # 청크 수와 같아야 함

벡터스토어 문서 수: 1835


##하이드 구현

###가상 문서 생성 단계

In [ ]:
def generate_hypothetical_doc(question: str) -> str:
    """질문에 대한 가상의 이상적인 답변 문서 생성"""
    prompt = f"""주어진 질문에 대해, 마치 실제 문서에서 발췌한 것 같은 이상적인 답변을 작성해주세요.
    단, 구체적인 수치, 날짜, 트렌드와 같은 상세 정보를 포함해야 합니다.

    질문: {question}

    답변:"""

    response = llm.invoke(prompt)
    return response.content


### 검색 과정

In [ ]:
def search_with_hyde(hypothetical_doc: str):
    """가상 문서를 이용해 실제 문서 검색"""
    results = vector_store.similarity_search_with_score(hypothetical_doc, k=4)
    return [
        {
            'content': doc.page_content,
            'score': score,
            'page': doc.metadata.get('page'),   # 추가: 페이지 번호
        } for doc, score in results
    ]

### 최종 답변 생성

In [ ]:
def generate_final_answer(question: str, relevant_docs: list) -> str:
    """검색된 문서를 바탕으로 최종 답변 생성"""
    context = "\n\n".join([doc['content'] for doc in relevant_docs])

    prompt = f"""다음 검색 결과를 바탕으로 질문에 답변해주세요.
검색 결과의 정보만 사용하고, 없는 정보는 답변하지 마세요.
답변은 핵심 내용 3가지를 번호 목록으로, 각 항목은 한두 문장으로 작성하세요.

검색 결과:
{context}

질문: {question}

답변:"""

    response = llm.invoke(prompt)
    return response.content

## 하이드 전체 프로세스 실행

In [ ]:
def process_query(question: str):
    """전체 HyDE 프로세스"""
    # 1. 가상 문서 생성
    print("1. 가상 문서 생성 중...")
    hypothetical_doc = generate_hypothetical_doc(question)
    print("\n가상 문서:", hypothetical_doc)

    # 2. 가상 문서로 검색
    print("\n2. 실제 문서 검색 중...")
    relevant_docs = search_with_hyde(hypothetical_doc)

    # 3. 최종 답변 생성
    print("\n3. 최종 답변 생성 중...")
    final_answer = generate_final_answer(question, relevant_docs)

    return {
        "hypothetical_doc": hypothetical_doc,
        "retrieved_docs": relevant_docs,
        "final_answer": final_answer
    }

## basic 파이프라인

In [ ]:
def process_query_basic(question: str):
    """HyDE 없이 기본 RAG 프로세스"""
    # 1. 질문으로 바로 검색
    print("1. 질문으로 문서 검색 중...")
    relevant_docs = search_with_hyde(question)   # 가상 문서 대신 질문을 그대로 넣음

    # 2. 최종 답변 생성
    print("\n2. 최종 답변 생성 중...")
    final_answer = generate_final_answer(question, relevant_docs)

    return {
        "retrieved_docs": relevant_docs,
        "final_answer": final_answer
    }

## HyDE 유무 비교

- 채점
judge(cross-encoder): 각 문서를 원래 질문과 짝지어 읽어 관련도 점수를
스코어링함   

In [ ]:
judge = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")

def judge_docs(question, docs):
    """원래 질문 기준으로 각 문서의 관련도 점수 (높을수록 관련)"""
    return judge.score([(question, d['content']) for d in docs])

questions = [
    "북한에서 강제로 이루어지는 조직 생활은 무엇이 있나요?",
    "북한 사람들은 어떤 단체에 의무적으로 들어가야 해?",
    "북한에서 공개처형은 어떤 경우에 이루어지나요?",
    # 질문만 적으면 됨, 정답 페이지 필요 없음
]

- 비교: 두 방법의 평균/최고 점수 비교

In [ ]:
rows = []
for q in questions:
    basic_docs = search_with_hyde(q)
    hyde_docs = search_with_hyde(generate_hypothetical_doc(q))
    bs, hs = judge_docs(q, basic_docs), judge_docs(q, hyde_docs)
    rows.append({
        '질문': q,
        'HyDE 전 평균': round(sum(bs) / len(bs), 3),
        'HyDE 후 평균': round(sum(hs) / len(hs), 3),
        'HyDE 전 최고': round(max(bs), 3),
        'HyDE 후 최고': round(max(hs), 3),
    })

df = pd.DataFrame(rows)
display(df)
print(df.drop(columns='질문').mean().round(3))

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the docum

,질문,HyDE 전 평균,HyDE 후 평균,HyDE 전 최고,HyDE 후 최고
0,북한에서 강제로 이루어지는 조직 생활은 무엇이 있나요?,0.001,0.001,0.003,0.003
1,북한 사람들은 어떤 단체에 의무적으로 들어가야 해?,0.059,0.253,0.092,0.941
2,북한에서 공개처형은 어떤 경우에 이루어지나요?,0.006,0.015,0.021,0.039


HyDE 전 평균    0.022
HyDE 후 평균    0.090
HyDE 전 최고    0.039
HyDE 후 최고    0.328
dtype: float32


차이가 많이 난 질문만 다시 실행 -> 가져온 문서 시각화

In [ ]:
q = "북한 사람들은 어떤 단체에 의무적으로 들어가야 해?"

basic_docs = search_with_hyde(q)
hypo = generate_hypothetical_doc(q)
hyde_docs = search_with_hyde(hypo)

print("[가상 문서]\n", ' '.join(hypo.split())[:300], "...\n")

for name, docs in [("HyDE 전", basic_docs), ("HyDE 후", hyde_docs)]:
    print(f"=== {name} ===")
    for d, s in zip(docs, judge_docs(q, docs)):
        print(f"[관련도 {s:.3f} | p.{d['page']}] {' '.join(d['content'].split())[:50]}")
    print()

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[가상 문서]
 북한의 사회 구조는 매우 복잡하고 체계화되어 있으며, 특정 단체에 대한 의무성이 강조되는 방식도 그 특징 중 하나입니다. 그러나 공식적인 자료나 문서에서 이러한 의무성에 대한 명확한 기록은 찾기 어렵습니다. 북한에서는 주로 국가사회주의 협동조합(NSSC)과 같은 조직이 존재하며, 이들 조직은 모든 북한 주민들에게 의무적으로 가입해야 합니다. 또한, 노동조합, 군대, 그리고 학교 등 다양한 조직에도 모두 가입해야 합니다. 그러나 이러한 의무성은 공식적으로 강조되지는 않으며, 주민들은 이러한 조직에 가입하지 않을 경우 사회적 비난을 받 ...

=== HyDE 전 ===
[관련도 0.068 | p.31] 당국의 행사와 각종 궐기대회에 집단적으로 동원되고 있으며, 동원 에 불참하면 당에 대한 충
[관련도 0.092 | p.208] 년단(이하 소년단), 사회주의애국청년동맹(이하 청년동맹), 조선직 업총동맹(이하 직맹), 
[관련도 0.000 | p.207] 2023 북한인권보고서 206 체가 동원되거나, 특정 회사가 지정되 면 그 조직 에 소속된
[관련도 0.077 | p.31] 마, 영화, 음악을 접촉·보관·유포한 경우에는 더욱 강하게 처벌할 수 있도록 한 것으로 알

=== HyDE 후 ===
[관련도 0.068 | p.31] 당국의 행사와 각종 궐기대회에 집단적으로 동원되고 있으며, 동원 에 불참하면 당에 대한 충
[관련도 0.941 | p.86] 으로서 통상적인 의무를 구성하는 것이다. 북한의 직장과 사회조직 을 통해 건설현장 등에 주
[관련도 0.001 | p.207] 고 있다. 2014년 북한인권조사위원회는 북한의 단체 중 사실상 국 가나 조선노동당의 통제
[관련도 0.000 | p.207] 2023 북한인권보고서 206 체가 동원되거나, 특정 회사가 지정되 면 그 조직 에 소속된



결과 해석:
**HyDE 전후 검색 결과 해석**

- **검색 결과 변화**: 문서 4개 중 2개가 바뀜. p.86(주민 동원은 시민의 통상적 의무가 아님)을 새로 찾았고, p.208(소년단·청년동맹 등 실제 조직 목록)을 놓침
- **장점**: 가상 문서가 질문 의도(의무적 조직 가입, 불참 시 비난)를 잘 담아, 표현이 다른 관련 문서(p.86)를 찾아냄 → 리랭커 최고점 0.092 → 0.941
- **할루시네이션 영향 적음**: 가상 문서에 가짜 조직명(NSSC)이 있었지만 엉뚱한 문서를 가져오진 않음. 세부 사실보다 **주제와 맥락**이 검색에 반영됨
- **한계**: LLM이 실제 북한 조직명을 몰라 가상 문서에 넣지 못했고, 그래서 가장 직접적인 답(p.208)을 놓침. **가상 문서에 빠진 정보는 검색에서도 빠짐** (도메인 지식 부족 문제)
- **평가 주의**: 리랭커 점수는 참고용. 핵심 문서(p.208)에 낮은 점수를 줘서, 실제 문서 확인이 필요함